# 03 — Hypothesis Testing

This notebook evaluates the Vanguard A/B test using completion rate, error rate, and session duration.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

pd.set_option("display.max_columns", None)


## Load clean data

In [ ]:

df = pd.read_csv("processed_data/clean_vanguard_data.csv")

df.head()


In [ ]:

print("Dataset shape:", df.shape)
display(df["variation"].value_counts())


## Prepare data

In [ ]:

df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")

step_mapping = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

df["process_step"] = df["process_step"].str.lower().str.strip()
df["step_num"] = df["process_step"].map(step_mapping)


## Create session-level data

In [ ]:

session_data = (
    df
    .groupby(["variation", "client_id", "visitor_id", "visit_id"], as_index=False)
    .agg(
        max_step=("step_num", "max"),
        first_timestamp=("date_time", "min"),
        last_timestamp=("date_time", "max"),
        number_of_steps=("process_step", "count")
    )
)

session_data["completed"] = session_data["max_step"] == 4

session_data["session_duration_seconds"] = (
    session_data["last_timestamp"] - session_data["first_timestamp"]
).dt.total_seconds()

session_data.head()



# Hypothesis Test 1 — Completion Rate

## Research Question

Did the new digital interface increase completion rate?

## Metric

Completion rate = completed sessions / total sessions

A session is completed when the user reaches the `confirm` step.

## Hypotheses

H₀: Completion rate is equal between control and test groups.

H₁: Completion rate is different between control and test groups.

Statistical test: two-proportion z-test.


In [ ]:

completion_summary = (
    session_data
    .groupby("variation")
    .agg(
        total_sessions=("visit_id", "count"),
        completed_sessions=("completed", "sum")
    )
    .reset_index()
)

completion_summary["completion_rate"] = (
    completion_summary["completed_sessions"] /
    completion_summary["total_sessions"]
)

completion_summary["completion_rate_percent"] = (
    completion_summary["completion_rate"] * 100
)

completion_summary


In [ ]:

plt.figure(figsize=(8, 5))

plt.bar(
    completion_summary["variation"],
    completion_summary["completion_rate_percent"]
)

plt.title("Completion Rate by Variation")
plt.xlabel("Variation")
plt.ylabel("Completion Rate (%)")
plt.ylim(0, 100)

plt.show()


In [ ]:

control = completion_summary[completion_summary["variation"] == "control"]
test = completion_summary[completion_summary["variation"] == "test"]

n_control = control["total_sessions"].values[0]
n_test = test["total_sessions"].values[0]

x_control = control["completed_sessions"].values[0]
x_test = test["completed_sessions"].values[0]

z_stat, p_value = proportions_ztest(
    [x_control, x_test],
    [n_control, n_test]
)

print("Control completed sessions:", x_control)
print("Control total sessions:", n_control)
print("Control completion rate:", round(x_control / n_control, 4))

print("Test completed sessions:", x_test)
print("Test total sessions:", n_test)
print("Test completion rate:", round(x_test / n_test, 4))

print("Z-statistic:", round(z_stat, 4))
print("P-value:", round(p_value, 4))


In [ ]:

alpha = 0.05

if p_value < alpha:
    print("Reject H0.")
    print("There is a statistically significant difference in completion rates.")
else:
    print("Fail to reject H0.")
    print("There is not enough evidence to say completion rates are significantly different.")


In [ ]:

control_rate = x_control / n_control
test_rate = x_test / n_test

absolute_lift = test_rate - control_rate
relative_lift = absolute_lift / control_rate

print("Control completion rate:", round(control_rate * 100, 2), "%")
print("Test completion rate:", round(test_rate * 100, 2), "%")
print("Absolute lift:", round(absolute_lift * 100, 2), "percentage points")
print("Relative lift:", round(relative_lift * 100, 2), "%")



## Completion Rate Conclusion

Use the p-value and lift above to decide whether the new interface created a meaningful improvement.

If p-value < 0.05, the difference is statistically significant.

If p-value >= 0.05, there is not enough statistical evidence to confirm a difference.



# Hypothesis Test 2 — Error Rate

## Research Question

Did the new interface reduce user errors?

## Metric

Error rate = backward movements / total actions

A backward movement happens when the user moves from a later step to an earlier step.

Examples:

- step_3 → step_2
- step_2 → step_1
- step_1 → start

Statistical test: two-proportion z-test.


In [ ]:

df_sorted = df.sort_values(
    by=["client_id", "visitor_id", "visit_id", "date_time"]
).copy()

df_sorted["previous_step_num"] = (
    df_sorted
    .groupby(["client_id", "visitor_id", "visit_id"])["step_num"]
    .shift(1)
)

df_sorted["is_error"] = df_sorted["step_num"] < df_sorted["previous_step_num"]

df_sorted[
    ["client_id", "visit_id", "process_step", "step_num", "previous_step_num", "is_error"]
].head()


In [ ]:

error_summary = (
    df_sorted
    .groupby("variation")
    .agg(
        total_actions=("process_step", "count"),
        total_errors=("is_error", "sum")
    )
    .reset_index()
)

error_summary["error_rate"] = (
    error_summary["total_errors"] /
    error_summary["total_actions"]
)

error_summary["error_rate_percent"] = error_summary["error_rate"] * 100

error_summary


In [ ]:

control_error = error_summary[error_summary["variation"] == "control"]
test_error = error_summary[error_summary["variation"] == "test"]

n_control_error = control_error["total_actions"].values[0]
n_test_error = test_error["total_actions"].values[0]

x_control_error = control_error["total_errors"].values[0]
x_test_error = test_error["total_errors"].values[0]

z_stat_error, p_value_error = proportions_ztest(
    [x_control_error, x_test_error],
    [n_control_error, n_test_error]
)

print("Control error rate:", round(x_control_error / n_control_error * 100, 2), "%")
print("Test error rate:", round(x_test_error / n_test_error * 100, 2), "%")
print("Z-statistic:", round(z_stat_error, 4))
print("P-value:", round(p_value_error, 4))


In [ ]:

if p_value_error < alpha:
    print("Reject H0.")
    print("There is a statistically significant difference in error rates.")
else:
    print("Fail to reject H0.")
    print("There is not enough evidence to say error rates are significantly different.")



# Hypothesis Test 3 — Session Duration

## Research Question

Did users in the test group complete sessions faster or slower?

## Metric

Session duration in seconds.

Statistical test: independent two-sample t-test.


In [ ]:

duration_data = session_data.copy()

duration_data = duration_data[
    duration_data["session_duration_seconds"].notna()
]

duration_data = duration_data[
    duration_data["session_duration_seconds"] >= 0
]

q1 = duration_data["session_duration_seconds"].quantile(0.25)
q3 = duration_data["session_duration_seconds"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

duration_no_outliers = duration_data[
    (duration_data["session_duration_seconds"] >= lower_bound) &
    (duration_data["session_duration_seconds"] <= upper_bound)
]

print("Original duration rows:", len(duration_data))
print("Rows after outlier removal:", len(duration_no_outliers))


In [ ]:

duration_summary = (
    duration_no_outliers
    .groupby("variation")
    .agg(
        average_duration=("session_duration_seconds", "mean"),
        median_duration=("session_duration_seconds", "median"),
        session_count=("session_duration_seconds", "count")
    )
    .reset_index()
)

duration_summary


In [ ]:

control_duration = duration_no_outliers[
    duration_no_outliers["variation"] == "control"
]["session_duration_seconds"]

test_duration = duration_no_outliers[
    duration_no_outliers["variation"] == "test"
]["session_duration_seconds"]

t_stat, p_value_duration = ttest_ind(
    control_duration,
    test_duration,
    equal_var=False
)

print("Control average duration:", round(control_duration.mean(), 2))
print("Test average duration:", round(test_duration.mean(), 2))
print("T-statistic:", round(t_stat, 4))
print("P-value:", round(p_value_duration, 4))


In [ ]:

if p_value_duration < alpha:
    print("Reject H0.")
    print("There is a statistically significant difference in average session duration.")
else:
    print("Fail to reject H0.")
    print("There is not enough evidence to say average session duration is significantly different.")



# Final Hypothesis Testing Summary

## Completion Rate

The completion rate test evaluates whether users in the test group reached the confirmation step more often than users in the control group.

## Error Rate

The error rate test evaluates whether one interface caused more backward movement than the other.

## Session Duration

The session duration test evaluates whether users spent significantly different amounts of time in the process.

## Business Interpretation

If the test group has:

- Higher completion rate
- Lower error rate
- Reasonable or reduced session duration

then the new interface may be considered more effective.

However, the final recommendation should depend on both statistical significance and business relevance.
                             


In [ ]:

alpha = 0.05

print("========== FINAL RESULTS ==========")

# Completion Rate
print("\nCOMPLETION RATE")
control_rate = x_control / n_control
test_rate = x_test / n_test
absolute_lift = test_rate - control_rate

print(f"Control: {round(control_rate*100,2)}%")
print(f"Test: {round(test_rate*100,2)}%")
print(f"Lift: {round(absolute_lift*100,2)} percentage points")
print(f"P-value: {round(p_value,4)}")

if p_value < alpha:
    print("Result: Statistically significant")
else:
    print("Result: NOT statistically significant")

# Error Rate
print("\nERROR RATE")
control_err_rate = x_control_error / n_control_error
test_err_rate = x_test_error / n_test_error

print(f"Control: {round(control_err_rate*100,2)}%")
print(f"Test: {round(test_err_rate*100,2)}%")
print(f"P-value: {round(p_value_error,4)}")

if p_value_error < alpha:
    print("Result: Statistically significant")
else:
    print("Result: NOT statistically significant")

# Session Duration
print("\nSESSION DURATION")
print(f"Control avg: {round(control_duration.mean(),2)} sec")
print(f"Test avg: {round(test_duration.mean(),2)} sec")
print(f"P-value: {round(p_value_duration,4)}")

if p_value_duration < alpha:
    print("Result: Statistically significant")
else:
    print("Result: NOT statistically significant")



# Final Conclusion

## Completion Rate
The test group completion rate is compared to the control group using a two-proportion z-test.

- If the p-value is **less than 0.05**, the difference is statistically significant.
- If the p-value is **greater than or equal to 0.05**, the difference is not statistically significant.

## Error Rate
The error rate analysis evaluates whether the new interface reduces backward movements in the process.

A statistically significant result would indicate a real difference between the two designs.

## Session Duration
The session duration analysis compares how long users take to complete the process.

A t-test is used to determine whether the difference between groups is statistically significant.

## Final Decision

The new interface is considered:

- ✅ **VALID improvement** if:
  - Completion rate is higher AND statistically significant
  - Error rate does not significantly increase

- ❌ **NOT VALID** if:
  - Completion rate is not statistically significant
  - OR error rate worsens
  - OR results are inconclusive

👉 The final decision should be based on the numerical results printed above.
